# 5.3 Dataclasses & Enums

**Prerequisites:** 5.1 Python OOPs, 4.5 Type Hints for Functions  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- `@dataclass` — what it generates, and what that saves you
- `field()`, `default_factory`, and the mutable-default fix
- `frozen=True`, `slots=True`, `kw_only=True`, `__post_init__`
- Choosing between a plain class, dataclass, NamedTuple, TypedDict and dict
- `Enum`, `IntEnum`, `StrEnum`, `auto()` and `Flag`
- Why enums beat bare string constants
- Enums with `match`/`case`

---

## Part 1 — Dataclasses

> **Version note:** `dataclasses` arrived in **Python 3.7**. `slots=True` and `kw_only=True`
> were added in **3.10**. These notes predate all of it.

A large share of the classes you write in real software are **mostly data**: a config
object, an API response, a parsed record, a domain entity. For those, the `__init__`,
`__repr__` and `__eq__` you write by hand are pure boilerplate — mechanical, repetitive, and
easy to get subtly wrong.

`@dataclass` reads your **type annotations** and generates them.

### Syntax breakdown

```
@dataclass
class ApiConfig:
    base_url: str                    <- required; becomes an __init__ parameter
    timeout: float = 30.0            <- has a default
    retries: int = 3
     |         |     |
     |         |     +-- default value
     |         +-------- the annotation is what makes it a field
     +------------------ field name
```

The annotation is not decoration — **`@dataclass` uses it to find the fields.** A class
attribute with no annotation is not a field.

In [ ]:
from dataclasses import dataclass


# ---- By hand: ~20 lines of boilerplate ----
class ApiConfigManual:
    def __init__(self, base_url: str, timeout: float = 30.0, retries: int = 3) -> None:
        self.base_url = base_url
        self.timeout = timeout
        self.retries = retries

    def __repr__(self) -> str:
        return (f"ApiConfigManual(base_url={self.base_url!r}, "
                f"timeout={self.timeout!r}, retries={self.retries!r})")

    def __eq__(self, other) -> bool:
        if not isinstance(other, ApiConfigManual):
            return NotImplemented
        return ((self.base_url, self.timeout, self.retries)
                == (other.base_url, other.timeout, other.retries))


# ---- With @dataclass: 4 lines, same behaviour ----
@dataclass
class ApiConfig:
    base_url: str
    timeout: float = 30.0
    retries: int = 3


cfg = ApiConfig("https://api.example.com")
print("repr    :", cfg)
print("equality:", ApiConfig("https://x.com") == ApiConfig("https://x.com"))
print("fields  :", cfg.base_url, cfg.timeout, cfg.retries)

# It is a normal, mutable class - you can still assign
cfg.timeout = 5.0
print("mutated :", cfg)

# What was generated for you
generated = [m for m in ("__init__", "__repr__", "__eq__") if m in ApiConfig.__dict__]
print("\ngenerated:", generated)

from dataclasses import fields
print("field metadata:")
for f in fields(ApiConfig):
    type_name = getattr(f.type, "__name__", str(f.type))
    print(f"  {f.name:<10} {type_name:<8} default={f.default!r}")

# ⚠️ A class attribute with NO annotation is not a field
@dataclass
class Wrong:
    annotated: int = 1
    not_annotated = 2          # just a class attribute - NOT in __init__ or __eq__

print("\nfields of Wrong:", [f.name for f in fields(Wrong)])
print("Wrong() ->", Wrong(), "| not_annotated still readable:", Wrong().not_annotated)

### `field()` and the mutable-default fix

You met the mutable default trap in **2.7** and **4.1**. Dataclasses would hit it too —
so they refuse outright.

```python
@dataclass
class Job:
    tags: list[str] = []          # ValueError at class-definition time
```

Python raises **`ValueError: mutable default <class 'list'> for field tags is not allowed`**
before you can even create an instance. The fix is `field(default_factory=...)`, which calls
the factory **once per instance**.

| `field()` argument | Purpose |
|---|---|
| `default_factory=list` | Fresh mutable default per instance |
| `default=...` | A plain immutable default |
| `init=False` | Not a constructor parameter — set in `__post_init__` |
| `repr=False` | Hide from `__repr__` (secrets, big blobs) |
| `compare=False` | Exclude from `__eq__` / ordering |

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timezone


# ---- Dataclasses catch the mutable-default bug for you ----
try:
    @dataclass
    class BrokenJob:
        name: str
        tags: list[str] = []
except ValueError as exc:
    print("refused at class definition:", exc)


# ---- The fix, plus the other field() options ----
@dataclass
class Job:
    name: str
    tags: list[str] = field(default_factory=list)          # fresh list per instance
    env: dict[str, str] = field(default_factory=dict)
    api_key: str = field(default="", repr=False)           # kept out of logs
    attempts: int = field(default=0, compare=False)        # not part of equality
    job_id: str = field(init=False)                        # derived, not passed in

    def __post_init__(self) -> None:
        """Runs after the generated __init__ - for derived fields and validation."""
        self.job_id = f"{self.name}-{len(self.tags)}"
        if not self.name:
            raise ValueError("job name must not be empty")


a = Job("reindex", tags=["nightly"])
b = Job("compact")

a.tags.append("urgent")
print("\na.tags:", a.tags)
print("b.tags:", b.tags, " <- independent list, as it should be")

print("\nrepr hides api_key:", Job("sync", api_key="sk-secret-123"))
print("derived job_id     :", a.job_id)

# compare=False in action: attempts is ignored by ==
x, y = Job("sync"), Job("sync")
y.attempts = 99
print("\nequal despite attempts differing:", x == y)

# __post_init__ validation
try:
    Job("")
except ValueError as exc:
    print("validation:", exc)

### `frozen`, `slots`, `kw_only` and ordering

Four options that cover most real needs:

| Option | Effect | Use for |
|---|---|---|
| `frozen=True` | Immutable; generates `__hash__` | Config, value objects, dict keys |
| `slots=True` (3.10+) | Adds `__slots__` — less memory, no typos | High-volume records |
| `kw_only=True` (3.10+) | All fields keyword-only | Classes with many fields |
| `order=True` | Generates `__lt__`, `__le__`, `__gt__`, `__ge__` | Sortable records |

**`frozen=True` is the one to reach for most often.** An immutable object can be safely
shared, cached, used as a dict key and passed between threads — and `@dataclass` gives you
a correct `__hash__` automatically, which is exactly the `__eq__`/`__hash__` pairing from
**5.1** done for you.

In [ ]:
from dataclasses import dataclass, replace, asdict, astuple
import sys


# ---- frozen: immutable and hashable ----
@dataclass(frozen=True)
class CacheKey:
    endpoint: str
    user_id: int


k1 = CacheKey("/api/profile", 42)
k2 = CacheKey("/api/profile", 42)

cache = {k1: "cached response"}
print("hashable, so usable as a dict key:", cache[k2])
print("equal instances                  :", k1 == k2)

try:
    k1.user_id = 99
except Exception as exc:
    print("immutable:", type(exc).__name__, "-", exc)

# replace() builds a NEW instance with some fields changed
k3 = replace(k1, user_id=99)
print("\nreplace() ->", k3, "| original unchanged:", k1)


# ---- slots: memory, measured ----
@dataclass
class RecordPlain:
    ts: str
    level: str
    message: str


@dataclass(slots=True)
class RecordSlots:
    ts: str
    level: str
    message: str


p = RecordPlain("2024-01-01", "ERROR", "db timeout")
s = RecordSlots("2024-01-01", "ERROR", "db timeout")
size_p = sys.getsizeof(p) + sys.getsizeof(p.__dict__)
size_s = sys.getsizeof(s)
print(f"\nplain: {size_p} bytes | slots: {size_s} bytes | saved {size_p - size_s} each")


# ---- kw_only: forces self-documenting construction ----
@dataclass(kw_only=True)
class RetryPolicy:
    max_attempts: int = 3
    backoff_seconds: float = 1.0
    jitter: bool = True


print("\n", RetryPolicy(max_attempts=5, jitter=False))
try:
    RetryPolicy(5, 1.0, False)
except TypeError as exc:
    print("positional refused:", exc)


# ---- order: sortable records ----
@dataclass(order=True)
class Release:
    major: int
    minor: int
    patch: int


versions = [Release(1, 10, 0), Release(1, 2, 0), Release(2, 0, 0)]
print("\nsorted:", sorted(versions))


# ---- Conversion helpers ----
cfg = RetryPolicy(max_attempts=2)
print("\nasdict :", asdict(cfg))
print("astuple:", astuple(cfg))

### Which container should I use?

This is the question people actually have, and it deserves a straight answer.

| | Mutable | Typed | Methods | Hashable | Best for |
|---|---|---|---|---|---|
| **`dict`** | ✅ | ❌ | ❌ | ❌ | Genuinely dynamic keys; JSON in flight |
| **`TypedDict`** | ✅ | ✅ (checker only) | ❌ | ❌ | Annotating an existing dict shape — JSON payloads |
| **`NamedTuple`** | ❌ | ✅ | ✅ | ✅ | Small immutable records; tuple-compatible (see **2.3**) |
| **`@dataclass`** | ✅ | ✅ | ✅ | with `frozen` | **The default choice** for a data-holding class |
| **`@dataclass(frozen=True)`** | ❌ | ✅ | ✅ | ✅ | Value objects, config, cache keys |
| **Plain class** | ✅ | ✅ | ✅ | ✅ | Behaviour-heavy classes where data is secondary |

### A decision rule

1. Is it mostly **behaviour**? -> plain class.
2. Is it mostly **data**? -> `@dataclass`.
3. Does it need to be **immutable / hashable**? -> `@dataclass(frozen=True)`.
4. Must it *be* a tuple (unpacking, existing tuple API)? -> `NamedTuple`.
5. Is it a **dict you don't control** (a JSON payload)? -> `TypedDict`.

> **Also worth knowing:** `pydantic` is a third-party library with dataclass-like syntax that
> *does* validate types at run time. It is the standard choice for parsing untrusted input
> (FastAPI is built on it). Dataclasses do **not** validate — the annotations are hints, as
> always (see **4.5**).

In [ ]:
from dataclasses import dataclass
from typing import NamedTuple, TypedDict


class PointTuple(NamedTuple):
    x: float
    y: float


@dataclass
class PointData:
    x: float
    y: float


class PointDict(TypedDict):
    x: float
    y: float


nt = PointTuple(1.0, 2.0)
dc = PointData(1.0, 2.0)
td: PointDict = {"x": 1.0, "y": 2.0}

print("NamedTuple:", nt, "| unpacks:", (lambda a, b: f"{a},{b}")(*nt))
print("dataclass :", dc)
print("TypedDict :", td, "| really a dict:", isinstance(td, dict))

print("\nNamedTuple is a tuple :", isinstance(nt, tuple))
print("NamedTuple is hashable:", hash(nt) is not None)

try:
    hash(dc)
except TypeError as exc:
    print("plain dataclass hashable:", exc)

print("\nmutable? NamedTuple:", end=" ")
try:
    nt.x = 5
    print("yes")
except AttributeError:
    print("no")

dc.x = 5
print("mutable? dataclass :", "yes ->", dc)

# ⚠️ Dataclasses do NOT validate types
bad = PointData("not a number", None)
print("\nno validation:", bad, " <- annotations are hints, not checks")

---

## Part 2 — Enums

> **Version note:** `enum` arrived in **3.4**; `StrEnum` in **3.11**.

### The problem enums solve

Real code is full of magic strings:

```python
if order.status == "shipped": ...
order.status = "shiped"          # typo - no error, silently never matches
```

Nothing catches the typo. Nothing tells you what the valid values are. Nothing stops
someone writing `"SHIPPED"` in one module and `"shipped"` in another.

An **`Enum`** is a set of named constants that are:

- **Discoverable** — `list(OrderStatus)` tells you every valid value
- **Typo-proof** — `OrderStatus.SHIPED` is an `AttributeError`, immediately
- **Self-describing** — `repr()` shows the name, not just the value
- **Singletons** — comparable with `is`
- **Type-checkable** — `mypy` rejects a bad value

**Real-world use case:** order/job/request states, HTTP methods and status classes,
permission levels, feature flags, log levels, currency codes — anything with a fixed set of
valid values.

In [ ]:
from enum import Enum, IntEnum, StrEnum, Flag, auto


# ---- The magic-string problem ----
status = "shipped"
if status == "shiped":                       # typo - silently False forever
    print("never runs")
print("typo in a string comparison is invisible:", status == "shiped")


# ---- A plain Enum ----
class OrderStatus(Enum):
    PENDING = "pending"
    PAID = "paid"
    SHIPPED = "shipped"
    DELIVERED = "delivered"
    CANCELLED = "cancelled"


print("\nmember   :", OrderStatus.SHIPPED)
print("name     :", OrderStatus.SHIPPED.name)
print("value    :", OrderStatus.SHIPPED.value)
print("all      :", [s.name for s in OrderStatus])
print("lookup   :", OrderStatus("paid"))
print("by name  :", OrderStatus["PAID"])

try:
    OrderStatus.SHIPED
except AttributeError as exc:
    print("\ntypo caught immediately:", exc)

try:
    OrderStatus("shiped")
except ValueError as exc:
    print("bad value caught       :", exc)

# Members are singletons, so `is` works
print("\nidentity comparison:", OrderStatus.PAID is OrderStatus("paid"))


# ---- auto(): when the value does not matter ----
class JobState(Enum):
    QUEUED = auto()
    RUNNING = auto()
    SUCCEEDED = auto()
    FAILED = auto()

print("\nauto values:", {s.name: s.value for s in JobState})

In [ ]:
from enum import Enum, IntEnum, StrEnum, Flag, auto


# ---- IntEnum: compares as an int ----
class HttpStatus(IntEnum):
    OK = 200
    CREATED = 201
    BAD_REQUEST = 400
    NOT_FOUND = 404
    SERVER_ERROR = 500


code = HttpStatus.NOT_FOUND
print("IntEnum:", code, "| == 404:", code == 404, "| >= 400:", code >= 400)
print("arithmetic works:", code + 0)

def is_error(status: HttpStatus) -> bool:
    return status >= HttpStatus.BAD_REQUEST

print("is_error(404):", is_error(HttpStatus.NOT_FOUND))


# ---- StrEnum (3.11+): compares as a str ----
class LogLevel(StrEnum):
    DEBUG = "debug"
    INFO = "info"
    WARNING = "warning"
    ERROR = "error"


level = LogLevel.WARNING
print("\nStrEnum:", level, "| == 'warning':", level == "warning")
print("string methods work:", level.upper())
print("f-string           :", f"level={level}")
print("JSON-serialisable  :", __import__("json").dumps({"level": LogLevel.ERROR}))


# ---- Flag: combinable options ----
class Permission(Flag):
    READ = auto()
    WRITE = auto()
    DELETE = auto()
    ADMIN = READ | WRITE | DELETE


user = Permission.READ | Permission.WRITE
print("\nuser perms      :", user)
print("can read        :", Permission.READ in user)
print("can delete      :", Permission.DELETE in user)
print("admin has all   :", Permission.READ in Permission.ADMIN)
print("add delete      :", user | Permission.DELETE)
print("remove write    :", user & ~Permission.WRITE)


# ---- Enums can carry methods ----
class Currency(Enum):
    INR = ("₹", 2)
    USD = ("$", 2)
    JPY = ("¥", 0)

    def __init__(self, symbol: str, decimals: int) -> None:
        self.symbol = symbol
        self.decimals = decimals

    def format(self, amount: float) -> str:
        return f"{self.symbol}{amount:,.{self.decimals}f}"


print("\n", Currency.INR.format(1234.5))
print(Currency.JPY.format(1234.5))

### Enums and `match`/`case`

Enums pair naturally with structural pattern matching (**3.4**) — and they sidestep the
capture-pattern trap entirely, because an enum member is always a **dotted name**, which
Python treats as a value pattern rather than a capture.

That is a real safety benefit, not just style.

In [ ]:
from enum import Enum


class JobState(Enum):
    QUEUED = "queued"
    RUNNING = "running"
    SUCCEEDED = "succeeded"
    FAILED = "failed"


def next_action(state: JobState, attempts: int = 0) -> str:
    match state:
        case JobState.QUEUED:
            return "start the worker"
        case JobState.RUNNING:
            return "wait and poll"
        case JobState.SUCCEEDED:
            return "record the result and clean up"
        case JobState.FAILED if attempts < 3:
            return f"retry (attempt {attempts + 1})"
        case JobState.FAILED:
            return "give up and alert"


for state in JobState:
    print(f"  {state.name:<10} -> {next_action(state)}")

print(f"  {'FAILED':<10} -> {next_action(JobState.FAILED, attempts=3)}")

# A dotted name is a VALUE pattern - it compares, it never captures.
# Compare with the bare-name trap in 3.4.
print("\nEnum members are dotted names, so the 3.4 capture trap cannot happen here.")


# Exhaustiveness: a checker can warn if you miss a member
def label(state: JobState) -> str:
    match state:
        case JobState.QUEUED | JobState.RUNNING:
            return "in progress"
        case JobState.SUCCEEDED:
            return "done"
        case JobState.FAILED:
            return "error"

print("\nlabels:", {s.name: label(s) for s in JobState})

---

## Common Mistakes & Pitfalls

1. **Forgetting the annotation on a dataclass field.** `count = 0` with no `: int` is just a class attribute — not in `__init__`, `__repr__` or `__eq__`.
2. **Trying `tags: list[str] = []`.** Dataclasses refuse it outright. Use `field(default_factory=list)`.
3. **Expecting a dataclass to validate types.** It does not. `PointData('abc', None)` is accepted. Use `pydantic` if you need run-time validation.
4. **Putting a field with a default before one without.** `TypeError: non-default argument follows default argument` — same rule as functions (**4.1**). `kw_only=True` sidesteps it.
5. **Assuming a plain dataclass is hashable.** It defines `__eq__`, so `__hash__` becomes `None` (**5.1**). Use `frozen=True`.
6. **Thinking `frozen=True` makes contents immutable.** A frozen dataclass holding a list still lets you mutate the list (**2.3**).
7. **Comparing an `Enum` member to its raw value.** `OrderStatus.PAID == 'paid'` is `False` for a plain `Enum` — only `IntEnum`/`StrEnum` compare to their values.
8. **Using an `Enum` where the value must serialise.** A plain `Enum` is not JSON-serialisable; `StrEnum`/`IntEnum` are.

## Best Practices

- Reach for **`@dataclass` by default** for classes that are mostly data.
- Use **`frozen=True`** unless you have a concrete reason to mutate — it gives you hashability and thread-safety for free.
- Use `field(default_factory=...)` for every mutable default.
- Use `__post_init__` for validation and derived fields.
- Use `slots=True` for classes you will create in bulk.
- Use `kw_only=True` once a class has more than about four fields.
- Replace **every** magic string constant with an `Enum`.
- Prefer `StrEnum`/`IntEnum` when the value crosses a serialisation boundary (JSON, DB, HTTP); use plain `Enum` otherwise.
- Type your function signatures with the enum, not `str` — that is where the checking pays off.

## Practice Exercises

Try these before moving on.

1. Convert a plain class you wrote in **5.1** into a dataclass. How many lines disappeared?
2. Write a frozen `Money` dataclass with `currency` and `amount`, and use it as a dict key.
3. Write a `RetryPolicy` dataclass with `kw_only=True` and validation in `__post_init__`.
4. Show the `ValueError` from a mutable default, then fix it with `default_factory`.
5. Build an `OrderStatus` enum and a `transition(current, target)` function that rejects invalid transitions, using `match`/`case`.
6. Model file permissions with `Flag` and write `can(user_perms, required)`.
7. Compare `sys.getsizeof` for 10,000 instances with and without `slots=True`.
8. Take a function with three boolean parameters and replace them with a `Flag` enum. Which call site reads better?